# MT3 Guitar Transcription Server

**BEFORE YOU START:** Go to `Runtime → Change runtime type` and select **GPU (T4)**.

This notebook:
1. Installs MT3 (Google's Multi-task Music Transcription model)
2. Downloads the pre-trained checkpoint
3. Starts a FastAPI server
4. Exposes it publicly via ngrok

Run each cell in order. **Cell 4 (model init) takes ~3–5 minutes.**
After **Cell 6**, copy the ngrok URL and paste it into the ScaleUp app's Advanced Settings.

In [ ]:
#@title Cell 1 — Install dependencies (runs once, ~3 min) { display-mode: "form" }
%%capture cap
import subprocess, sys

# Core ML stack
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'note-seq==0.0.5',
    'seqio-nightly',
], check=True)

# MT3 from Google Research
subprocess.run(['git', 'clone', '--quiet',
    'https://github.com/google-research/mt3.git',
    '/content/mt3_repo'], check=False)  # ignore if already cloned

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '/content/mt3_repo',
], check=True)

# Server stack
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi',
    'uvicorn[standard]',
    'python-multipart',
    'pyngrok',
    'nest-asyncio',
    'librosa',
    'soundfile',
], check=True)

print('✅ All packages installed')

In [ ]:
#@title Cell 2 — Download MT3 checkpoint from Google Cloud (~800 MB) { display-mode: "form" }
import os

CHECKPOINT_DIR = '/content/mt3_checkpoints'

if os.path.isdir(os.path.join(CHECKPOINT_DIR, 'mt3')):
    print('✅ Checkpoint already downloaded')
else:
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print('Downloading MT3 checkpoint (this takes ~2 min)…')
    ret = os.system(f'gsutil -m cp -r gs://mt3/checkpoints/mt3 {CHECKPOINT_DIR}/')
    if ret != 0:
        # Fallback: try authenticated copy
        print('Trying with gcloud auth…')
        os.system('gcloud auth application-default login --no-launch-browser')
        os.system(f'gsutil -m cp -r gs://mt3/checkpoints/mt3 {CHECKPOINT_DIR}/')
    print('✅ Checkpoint downloaded')

# Show what we have
for root, dirs, files in os.walk(CHECKPOINT_DIR):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) // 1024
        print(f'  {path} ({size} KB)')

In [ ]:
#@title Cell 3 — Configure ngrok auth token { display-mode: "form" }
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = ''  #@param {type: "string"}

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok as _ngrok
    _ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print('✅ ngrok auth token set')
else:
    print('⚠️  No token — ngrok will work for ~2 hrs per session without auth.')
    print('   Sign up free at https://ngrok.com to remove the limit.')

In [ ]:
#@title Cell 4 — Load MT3 model (~3–5 min on first run) { display-mode: "form" }
import sys, os
sys.path.insert(0, '/content/mt3_repo')

import functools
import numpy as np
import note_seq
import seqio

from mt3 import audio_utils
from mt3 import event_codec
from mt3 import metrics_utils
from mt3 import models as mt3_models
from mt3 import network as mt3_network
from mt3 import note_sequences
from mt3 import preprocessors
from mt3 import run_length_encoding
from mt3 import spectrograms as mt3_spectrograms
from mt3 import vocabularies

import t5x
import jax

print(f'JAX devices: {jax.devices()}')

# ── Codec & spectrogram config (must match checkpoint) ────────────────────────
SAMPLE_RATE     = audio_utils.SAMPLE_RATE   # 16 000 Hz
HOP_SIZE        = 128
FRAMES_PER_SEC  = SAMPLE_RATE // HOP_SIZE   # 125

CODEC = vocabularies.build_codec(
    note_sequences.NoteEventData,
    audio_sample_rate=SAMPLE_RATE,
)
VOCABULARY = vocabularies.vocabulary_from_codec(CODEC)

SPEC_CONFIG = mt3_spectrograms.SpectrogramConfig(
    sample_rate=SAMPLE_RATE,
    spec_hop_length=HOP_SIZE,
    num_mel_bins=512,
    mel_lo_hz=50.0,
    mel_hi_hz=8000.0,
)

# ── Model architecture ────────────────────────────────────────────────────────
MODEL_CONFIG = mt3_network.T5Config(
    vocab_size=VOCABULARY.vocab_size,
    dtype='bfloat16',
    emb_dim=512,
    num_heads=6,
    num_encoder_layers=8,
    num_decoder_layers=8,
    head_dim=64,
    mlp_dim=1024,
    mlp_activations=('gelu',),
    dropout_rate=0.0,
    logits_via_embedding=False,
)

MODEL = mt3_models.ContinuousInputsEncoderDecoderModel(
    module=mt3_network.Transformer(config=MODEL_CONFIG),
    input_vocabulary=VOCABULARY,
    output_vocabulary=VOCABULARY,
    optimizer_def=None,
    input_depth=SPEC_CONFIG.num_mel_bins,
)

# ── Load checkpoint ───────────────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'mt3')
print('Loading checkpoint…')
TRAIN_STATE = t5x.checkpoints.load_t5x_checkpoint(
    CHECKPOINT_PATH,
    step=None,
    state_transformation_fns=[],
)
print('✅ Model loaded!')

# ── Build inference function (JIT-compiled) ───────────────────────────────────
_partitioner = t5x.partitioning.PjitPartitioner(num_partitions=1)
_model_params = TRAIN_STATE['target']

@functools.lru_cache(maxsize=None)
def _get_infer_step():
    """Returns a JIT-compiled inference step."""
    def _infer(params, batch):
        return MODEL.predict_batch_with_aux(params, batch)
    return jax.jit(_infer)


def transcribe_audio(audio_path: str) -> note_seq.NoteSequence:
    """Runs MT3 on an audio file and returns a NoteSequence."""
    import librosa

    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    # Split into overlapping 5-second segments
    segment_length = SAMPLE_RATE * 5
    hop            = SAMPLE_RATE * 4   # 1 s overlap
    segments       = []
    start          = 0
    while start < len(audio):
        seg = audio[start : start + segment_length]
        if len(seg) < segment_length:
            seg = np.pad(seg, (0, segment_length - len(seg)))
        segments.append((start / SAMPLE_RATE, seg))
        start += hop
        if start >= len(audio):
            break

    ns_combined = note_seq.NoteSequence()
    infer_fn    = _get_infer_step()

    for seg_start_s, seg_audio in segments:
        # Compute spectrogram
        spec = mt3_spectrograms.compute_spectrogram(seg_audio, SPEC_CONFIG)
        spec = spec[np.newaxis]  # batch dim

        batch = {
            'encoder_input_tokens':  spec,
            'decoder_input_tokens':  np.zeros((1, 1024), dtype=np.int32),
        }

        output = infer_fn(_model_params, batch)
        token_ids = np.array(output[0][0])  # [seq_len]

        # Decode tokens → NoteSequence events
        try:
            events = VOCABULARY.decode(token_ids.tolist())
            ns_seg = run_length_encoding.decode_events(
                state=note_sequences.NoteEncodingState(),
                tokens=events,
                start_time=seg_start_s,
                max_time=seg_start_s + 5.0,
                codec=CODEC,
                decode_event_fn=note_sequences.decode_note_event,
            )
            for note in ns_seg.notes:
                ns_combined.notes.append(note)
        except Exception as decode_err:
            print(f'[MT3] decode error at {seg_start_s:.1f}s: {decode_err}')
            continue

    ns_combined.total_time = len(audio) / SAMPLE_RATE
    return ns_combined


def note_sequence_to_json(ns: note_seq.NoteSequence) -> list:
    """Converts NoteSequence to the DetectedNote JSON format expected by ScaleUp."""
    results = []
    for note in ns.notes:
        midi   = int(note.pitch)
        freq   = 440.0 * (2 ** ((midi - 69) / 12.0))
        conf   = note.velocity / 127.0  # normalise velocity → 0-1 confidence
        results.append({
            'startTime':  round(float(note.start_time), 4),
            'endTime':    round(float(note.end_time),   4),
            'midiNote':   midi,
            'confidence': round(conf, 3),
            'frequency':  round(freq, 3),
        })
    results.sort(key=lambda n: n['startTime'])
    return results


print('✅ Inference helpers ready!')

In [ ]:
#@title Cell 5 — Quick sanity test (optional but recommended) { display-mode: "form" }
import tempfile, soundfile as sf, numpy as np

# Generate a 2-second 440 Hz sine wave (A4 = MIDI 69)
t    = np.linspace(0, 2, 2 * SAMPLE_RATE)
sine = (0.3 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
    sf.write(f.name, sine, SAMPLE_RATE)
    tmp_path = f.name

ns_test = transcribe_audio(tmp_path)
notes   = note_sequence_to_json(ns_test)
print(f'Detected {len(notes)} notes (expect at least 1 near MIDI 69):')
for n in notes:
    print(f"  MIDI {n['midiNote']} @ {n['startTime']:.2f}s–{n['endTime']:.2f}s  conf={n['confidence']:.2f}")

In [ ]:
#@title Cell 6 — Start FastAPI + ngrok server { display-mode: "form" }
#
# After running this cell, copy the printed ngrok URL and paste it into
# ScaleUp → Audio to Tab → ⚙️ Advanced → MT3 Server URL
#
# The server stays alive as long as this cell is running.
# To stop: Runtime → Interrupt execution

import asyncio, json, os, tempfile
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title='MT3 Guitar Transcription', version='1.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['GET', 'POST', 'OPTIONS'],
    allow_headers=['*'],
)


@app.get('/health')
def health():
    """Health check — use to verify the server is reachable."""
    return {'status': 'ok', 'model': 'mt3'}


@app.post('/transcribe')
async def transcribe(audio: UploadFile = File(...)):
    """
    Accepts a WAV, MP3, OGG, M4A or WEBM audio file.
    Returns JSON: { "notes": [{startTime, endTime, midiNote, confidence, frequency}], "duration": float }
    """
    # Validate MIME / extension
    filename    = audio.filename or 'audio.wav'
    ext         = os.path.splitext(filename)[-1].lower()
    allowed_ext = {'.wav', '.mp3', '.ogg', '.m4a', '.webm', '.flac'}
    if ext not in allowed_ext:
        raise HTTPException(400, f'Unsupported format: {ext}. Use: {allowed_ext}')

    # Save to temp file
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        data = await audio.read()
        tmp.write(data)
        tmp_path = tmp.name

    try:
        ns      = transcribe_audio(tmp_path)
        notes   = note_sequence_to_json(ns)
        dur     = round(float(ns.total_time), 3)
        return {'notes': notes, 'duration': dur, 'count': len(notes)}
    except Exception as e:
        raise HTTPException(500, f'Transcription failed: {e}') from e
    finally:
        os.unlink(tmp_path)


# ── Launch ────────────────────────────────────────────────────────────────────
PORT = 8080
ngrok_tunnel = ngrok.connect(PORT)
public_url   = ngrok_tunnel.public_url

print('=' * 60)
print(f'🎸 MT3 Transcription Server is LIVE!')
print(f'   Public URL:  {public_url}')
print(f'   Health:      {public_url}/health')
print(f'   Transcribe:  POST {public_url}/transcribe')
print()
print('Paste this URL into ScaleUp → Audio to Tab → ⚙️ Advanced:')
print(f'  {public_url}')
print('=' * 60)

uvicorn.run(app, host='0.0.0.0', port=PORT)